# Game Oracle — Data Collection Example

This notebook demonstrates how to use the Game Oracle data collection module to fetch and process Steam game data.

## Setup

Before running this notebook, ensure:
1. Dependencies are installed: `pip install -r ../backend/requirements.txt`
2. You're in the correct directory
3. Optional: Set up `.env` file with Steam API key (if needed)

In [1]:
# Add backend module to path
import sys
from pathlib import Path

backend_path = Path('../backend').resolve()
sys.path.insert(0, str(backend_path))

print(f"Backend path: {backend_path}")
print(f"Path exists: {backend_path.exists()}")

Backend path: D:\PROJELER\SteamAnalysisTool\backend
Path exists: True


In [ ]:
# Configure logging
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

# Reduce noise from urllib3
logging.getLogger('urllib3').setLevel(logging.WARNING)

In [2]:
# Import the data collection module
from data_collection import (
    GameOraclePipeline,
    TextCleaner,
    SteamAPIClient,
    SteamSpyClient,
)

print("✓ Successfully imported Game Oracle modules")

✓ Successfully imported Game Oracle modules


## 1. Initialize the Pipeline

In [3]:
# Create pipeline instance
pipeline = GameOraclePipeline()

print("✓ Pipeline initialized")
print(f"  - Steam client: {type(pipeline.steam_client).__name__}")
print(f"  - SteamSpy client: {type(pipeline.steamspy_client).__name__}")
print(f"  - Data processor: {type(pipeline.data_processor).__name__}")

✓ Pipeline initialized
  - Steam client: SteamAPIClient
  - SteamSpy client: SteamSpyClient
  - Data processor: DataProcessor


## 2. Text Cleaning Example

In [4]:
# Example of raw Steam review text
raw_review = """
<p>This game is <b>absolutely amazing</b>!</p>
Great gameplay and storyline.\n\nCheck out my channel: http://youtube.com/user123
[h1]Highly Recommended[/h1]
&quot;Must play&quot; - IGN
"""

print("Raw review:")
print(raw_review)
print("\n" + "="*60 + "\n")

# Clean it
cleaned = TextCleaner.clean_review(raw_review)

print("Cleaned review:")
print(cleaned)

Raw review:

<p>This game is <b>absolutely amazing</b>!</p>
Great gameplay and storyline.

Check out my channel: http://youtube.com/user123
[h1]Highly Recommended[/h1]
&quot;Must play&quot; - IGN



Cleaned review:
This game is absolutely amazing! Great gameplay and storyline. Check out my channel: http://youtube.com/user123 Highly Recommended "Must play" - IGN


## 3. Test with a Single Game

Let's fetch data for Counter-Strike 2 (app_id: 730) — a well-known game with plenty of reviews.

In [ ]:
# Process a single well-known game
print("Fetching data for Counter-Strike 2...\n")

reviews_df, summary_df = pipeline.process_single_game(
    title="Counter-Strike 2",
    max_reviews=100  # Small number for demo
)

print(f"\n✓ Completed!")
print(f"  - Reviews collected: {len(reviews_df)}")
print(f"  - Summary rows: {len(summary_df)}")

## 4. Explore the Reviews DataFrame

In [ ]:
# Display basic info
print(f"DataFrame shape: {reviews_df.shape}")
print(f"\nColumns: {list(reviews_df.columns)}")

In [ ]:
# Show first few reviews
display_cols = ['name', 'review_text', 'sentiment_label', 'sentiment_score']
reviews_df[display_cols].head(5)

In [ ]:
# Sentiment distribution
print("Sentiment Distribution:")
print(reviews_df['sentiment_label'].value_counts())
print(f"\nPositive ratio: {(reviews_df['sentiment_score'].sum() / len(reviews_df)):.1%}")

In [ ]:
# Show data types and missing values
print("Data Info:")
print(f"\nData types:")
print(reviews_df.dtypes)

print(f"\nMissing values:")
print(reviews_df.isna().sum())

## 5. Explore the Summary DataFrame

In [ ]:
# Show summary data
print("Game Summary:")
summary_df

In [ ]:
# Extract market insights
if not summary_df.empty:
    game = summary_df.iloc[0]
    print(f"Game: {game['name']}")
    print(f"Price: ${game['price_usd']:.2f}")
    print(f"Estimated Players: {game['estimated_owners_min']:,} - {game['estimated_owners_max']:,}")
    print(f"Positive Review Ratio: {game['positive_ratio']:.1%}")
    print(f"Total Reviews: {game['total_reviews']:,}")
    print(f"Genres: {game['genres']}")
    print(f"Top Tags: {game['tags']}")

## 6. Batch Processing Multiple Games

⚠️ **Note:** This will make many API calls. Be patient and respect rate limits.

In [ ]:
# Example games (adjust as needed)
games_to_analyze = [
    "Counter-Strike 2",
    "Dota 2",
    "PUBG: Battlegrounds",
]

print(f"Analyzing {len(games_to_analyze)} games...\n")

# This will take a while due to rate limiting
# Uncomment to run:

# all_reviews_df, all_summary_df = pipeline.collect_and_process(
#     game_titles=games_to_analyze,
#     max_reviews_per_game=200,  # Reduced for demo
#     save_path=None  # Set to "data/output" to save
# )

# print(f"\n✓ Completed!")
# print(f"Total reviews: {len(all_reviews_df)}")
# print(f"Total games: {len(all_summary_df)}")

## 7. Export Data for Analysis

In [ ]:
# Export reviews for NLP fine-tuning
# This format is perfect for transformer models

nlp_format = reviews_df[['review_text', 'sentiment_score']].copy()
nlp_format = nlp_format.rename(columns={
    'review_text': 'text',
    'sentiment_score': 'label'
})

print(f"NLP format ready for fine-tuning:")
print(nlp_format.head())

# Save it
# nlp_format.to_csv('reviews_for_nlp.csv', index=False)
# print(f"\n✓ Saved to reviews_for_nlp.csv")

## 8. Data Quality Checks

In [ ]:
# Validate data quality
processor = pipeline.data_processor

is_valid = processor.validate_dataframe(reviews_df)
print(f"DataFrame validation passed: {is_valid}")

In [ ]:
# Check text length distribution
reviews_df['text_length'] = reviews_df['review_text'].str.len()

print(f"Review text length statistics:")
print(reviews_df['text_length'].describe())
print(f"\nMin length: {reviews_df['text_length'].min()}")
print(f"Max length: {reviews_df['text_length'].max()}")

## Next Steps

Now that you have AI-ready data, you can:

1. **Fine-tune NLP Models**: Use the `reviews_df` with columns `review_text` and `sentiment_score` to fine-tune BERT, DistilBERT, or other transformers

2. **Market Analysis**: Use the `summary_df` for:
   - Price vs. Review Sentiment analysis
   - Market Size Estimation
   - Genre/Tag Performance
   - Regional Release Date Impact

3. **Feature Engineering**: Create derived features:
   - Review velocity (reviews over time)
   - Sentiment trend
   - Price elasticity

See other notebooks in this directory for advanced analysis!